# Hold the disclosure: real pilot (Tiny Aya)

Runs the pre-registered pilot on Kaggle. Three arms, the same 60 dialogues (20 per script, matched across scripts), greedy decoding:

1. `CohereLabs/tiny-aya-fire` with the plain prompt
2. `CohereLabs/tiny-aya-fire` with the strict prompt
3. `CohereLabs/tiny-aya-global` with the plain prompt

Outputs: transcripts, `summary.json` (DR@k, Kept@5 given T1, bootstrap CI), `decision.json` (the go/no-go rule from `docs/PREREGISTRATION.md`).

Needs: a Kaggle secret named `HF_TOKEN`. Tiny Aya is gated, so the account must have accepted the licence.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted. input: " + str(sorted(str(p) for p in Path('/kaggle/input').glob('*'))))
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
SRC = find_repo()
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("code root:", SRC)

def find_dir(marker, parent_depth):
    hits = [p for p in Path("/kaggle/input").rglob(marker) if p.exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit(f"{marker} not mounted")
    p = hits[0]
    for _ in range(parent_depth):
        p = p.parent
    return p

def read_token():
    for p in Path("/kaggle/input").rglob("hf_token.txt"):
        return p.read_text().strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

if not os.environ.get("HF_TOKEN"):
    tok = read_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ.setdefault("HUGGINGFACE_API_KEY", tok)
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets (private) or add a Kaggle secret HF_TOKEN"

FIRE = "CohereLabs/tiny-aya-fire"
GLOBAL = "CohereLabs/tiny-aya-global"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)


In [ ]:
run([sys.executable, "scripts/build_pilot.py", "--split", "pilot", "--n-per-script", "20", "--out", "data/generated/pilot_dialogues.jsonl"])

In [ ]:
ARMS = [(FIRE, "plain", "fire_plain"), (FIRE, "strict", "fire_strict"), (GLOBAL, "plain", "global_plain")]
for model, variant, tag in ARMS:
    run([sys.executable, "scripts/run_pilot.py", "--dialogues", "data/generated/pilot_dialogues.jsonl",
         "--model", model, "--variant", variant, "--backend", "hf", "--device", "cuda",
         "--batch-size", "8", "--max-new-tokens", "384", "--out", f"outputs/pilot/{tag}.jsonl"])

In [ ]:
run([sys.executable, "scripts/score_run.py", "outputs/pilot/fire_plain.jsonl", "outputs/pilot/fire_strict.jsonl", "outputs/pilot/global_plain.jsonl", "--out", "outputs/pilot/summary.json"])
run([sys.executable, "scripts/pilot_decision.py", "outputs/pilot/summary.json", "--out", "outputs/pilot/decision.json"])

In [ ]:
from htd.replay import load_transcripts

for tag in ("fire_plain", "global_plain"):
    ts = load_transcripts(f"outputs/pilot/{tag}.jsonl")
    print(f"\n########## {tag}: turn 5 reply, one dialogue per script")
    for script in ("en", "hi", "hinglish"):
        t = next(x for x in ts if x.script == script)
        print(f"\n--- {script} | {t.product_id} | {t.dialogue_id}")
        print("CUSTOMER:", t.user_turns[4])
        print("MODEL:", t.replies[4])

In [ ]:
import zipfile

ART = Path("/kaggle/working/htd-artifacts")
if ART.exists():
    shutil.rmtree(ART)
shutil.copytree(WORK / "outputs", ART / "outputs")
shutil.copytree(WORK / "data" / "generated", ART / "generated")
with zipfile.ZipFile("/kaggle/working/htd-pilot-artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in ART.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(ART))
print("wrote /kaggle/working/htd-pilot-artifacts.zip")
print(sorted(str(p.relative_to(ART)) for p in ART.rglob("*") if p.is_file()))